<table width="100%">
<tr>
<td width="50%" align="left"><a href="./00_EDA.ipynb">← Previous: EDA</a></td>
<td width="50%" align="right"><a href="./02_RQ2_capacity_workforce.ipynb">Next: RQ2 — Capacity Workforce →</a></td>
</tr>
</table>

## **1.0 Population Size and Demographic Composition Changes**

#### In this section, we would look at how Population size and demographic composition of Canadian provinces and territories have evolved as well as the implication of this changes on healthcare demand

> - #### **Research Quesiton 1: How have population size and demographic composition changed across Canadian provinces and territories, and what implications do these changes have for healthcare demand?**

In [ ]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
from dotenv import load_dotenv

import os

import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
from daytascape_db_core.connection import get_db_engine

In [ ]:
DATABASE = "health_system_performance"

engine = get_db_engine(DATABASE)

print("Connected database:", engine.url.database)

In [ ]:
query = """
SELECT *
FROM master.analytics_province_year;
"""

df = pd.read_sql(query, engine)

print("Connected successfully to database!")

df.head(10)

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

In [ ]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

In [ ]:
df.isna().sum()

In [ ]:
query = """
    SELECT
    province_id,
    province_code,
    province_name,
    data_year,
    population,
    population_65_plus,
    population_80_plus,
    population_85_plus,
    population_65_share_pct,
    population_80_share_pct,
    population_85_share_pct
FROM master.analytics_province_year
WHERE province_code NOT IN ('NT', 'YT', 'NU')
ORDER BY province_name, data_year;
"""

df_population_demand = pd.read_sql(query, engine)

df_population_demand.head()

### **1.1 Population Growth**

In [ ]:
df_population_demand["population_growth_pct"] = (
    df_population_demand
    .groupby('province_code')["population"]
    .pct_change() * 100
)

df_population_demand

In [ ]:
df_population_demand["population_65_plus_growth_pct"] = (
df_population_demand
.groupby("province_code")["population_65_plus"]
.pct_change() * 100
)

In [ ]:
df_population_demand["population_80_plus_growth_pct"] = (
df_population_demand
.groupby("province_code")["population_80_plus"]
.pct_change() * 100
)

In [ ]:
df_population_demand["population_85_plus_growth_pct"] = (
df_population_demand
.groupby("province_code")["population_85_plus"]
.pct_change() * 100
)


In [ ]:
df_population_demand[
    [
        "province_name",
        "province_code",
        "population",
        "population_growth_pct"
    ]
    
].head()

In [ ]:
df_population_demand["population_growth_pct"].describe()

### **1.2 Population Change over the period 1971-2025**

In [ ]:
df_population_change = (
    df_population_demand
    .sort_values(["province_code", "data_year"])
    .groupby(["province_code", "province_name"])
    .agg(
        start_year=("data_year", "min"),
        end_year=("data_year", "max"),
        start_population=("population", "first"),
        end_population=("population", "last")
        
    )
    .reset_index()
)

df_population_change

In [ ]:
df_population_change["absolute_change"] =(
    df_population_change["end_population"] - 
    df_population_change["start_population"]
)

In [ ]:
df_population_change["population_change_pct"] = (
    df_population_change["absolute_change"]
    / df_population_change["start_population"]
) * 100

In [ ]:

df_population_change

### **1.3 Figures**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,7))

for province, group in df_population_demand.groupby("province_name"):
    plt.plot(
        group["data_year"],
        group["population"],
        label=province
    )   

plt.title("Population Trends Across Canadian Provinces (1971-2025)")
plt.xlabel("Year")
plt.ylabel("Population")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
df_population_change.sort_values(
    "absolute_change", ascending=False
)

In [ ]:
df_population_change.sort_values(
    "population_change_pct",
    ascending=False
)

### **1.4 Population Ageing Demography**

In [ ]:
df_population_aging = df_population_demand[
    [
        "province_code",
        "province_name",
        "data_year",
        "population_65_plus",
        "population_80_plus",
        "population_85_plus",
        "population_65_share_pct",
        "population_80_share_pct",
        "population_85_share_pct"
    ]
].copy()

In [ ]:
df_population_aging

In [ ]:
df_population_demand["growth_65_plus_pct"] = (
    df_population_demand
    .groupby("province_code")["population_65_plus"]
    .pct_change() * 100
)

In [ ]:
df_population_demand["growth_80_plus_pct"] = (
df_population_demand
.groupby("province_code")["population_80_plus"]
.pct_change() * 100
)

In [ ]:
df_population_demand["growth_85_plus_pct"] = (
df_population_demand
.groupby("province_code")["population_85_plus"]
.pct_change() * 100
)

In [ ]:
df_population_demand

In [ ]:
plt.figure(figsize=(12,7))
for province, group in df_population_demand.groupby("province_name"):
    plt.plot(
        group["data_year"],
        group["population_65_share_pct"],
        label=province
    )

plt.title("Population Aged 65+ as a Share of Total Population (1971-2025)")
plt.xlabel("Year")
plt.ylabel("Population 65+ (%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,7))
for province, group in df_population_demand.groupby("province_name"):
    plt.plot(
        group["data_year"],
        group["population_80_share_pct"],
        label=province
    )

plt.title("Population Aged 80+ as a Share of Total Population (1971-2025)")
plt.xlabel("Year")
plt.ylabel("Population 65+ (%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,7))
for province, group in df_population_demand.groupby("province_name"):
    plt.plot(
        group["data_year"],
        group["population_85_share_pct"],
        label=province
    )

plt.title("Population Aged 85+ as a Share of Total Population (1971-2025)")
plt.xlabel("Year")
plt.ylabel("Population 65+ (%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### **1.5 -  Ageing Population Average Growth Across Provinces (1970-2024)**

#### **1.5.1 -  65+ Ageing Population Average Growth (1971-2024)**

In [ ]:
df_population_demand.groupby(
    ["province_code", "province_name"]
)["growth_65_plus_pct"].mean().sort_values(ascending=False)

#### **1.5.2 - 80+ Ageing Population Average Growth (1971-2024)**

In [ ]:
df_population_demand.groupby(
    ["province_code", "province_name"]
)["growth_80_plus_pct"].mean().sort_values(ascending=False)

#### **1.5.3 - 85+ Ageing Population Average Growth (1971-2024)**

In [ ]:
df_population_demand.groupby(
    ["province_code", "province_name"]
)["growth_85_plus_pct"].mean().sort_values(ascending=False)

### **1.6 - Share of Ageing Population Across Province**

#### **1.6.1 - 65+ Share of Population**

In [ ]:
df_population_demand.groupby(
    ["province_code", "province_name"]
)["population_65_share_pct"].mean().sort_values(ascending=False)

#### **1.6.2 - 80+ Share of Population**

In [ ]:
df_population_demand.groupby(
    ["province_code", "province_name"]
)["population_80_share_pct"].mean().sort_values(ascending=False)

#### **1.6.3 - 85+ Share of Population**

In [ ]:
df_population_demand.groupby(
    ["province_code", "province_name"]
)["population_85_share_pct"].mean().sort_values(ascending=False)

In [ ]:
plot_df = (
    df_population_demand
    .groupby(["province_code", "province_name"])[
        [
            "population_growth_pct",
            "growth_65_plus_pct",
            "growth_80_plus_pct",
            "growth_85_plus_pct"
        ]
    ]
    .mean()
    .reset_index()
)

# Separate Canada from the provinces
canada = plot_df[plot_df["province_code"] == "CAN"]
provinces = plot_df[plot_df["province_code"] != "CAN"].copy()

# Sort provinces by 65+ growth
provinces = provinces.sort_values("growth_65_plus_pct")

# Add Canada as the benchmark
plot_df = pd.concat([provinces, canada], ignore_index=True)

# Plot
y = np.arange(len(plot_df))
height = 0.20

fig, ax = plt.subplots(figsize=(11, 6.5))

ax.barh(
    y - 1.5 * height,
    plot_df["population_growth_pct"],
    height,
    label="Total population"
)

ax.barh(
    y - 0.5 * height,
    plot_df["growth_65_plus_pct"],
    height,
    label="Age 65+"
)

ax.barh(
    y + 0.5 * height,
    plot_df["growth_80_plus_pct"],
    height,
    label="Age 80+"
)

ax.barh(
    y + 1.5 * height,
    plot_df["growth_85_plus_pct"],
    height,
    label="Age 85+"
)

ax.set_yticks(y)
ax.set_yticklabels(plot_df["province_name"])

ax.set_xlabel("Average Annual Growth Rate (%)")

ax.set_title(
    "Population Growth and Demographic Aging Across Canada",
    fontsize=14,
    fontweight="bold"
)

ax.legend(
    frameon=False,
    ncol=4,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08)
)

ax.grid(
    axis="x",
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
df_population_demand

In [ ]:
canada = (
    df_population_demand[
        df_population_demand["province_code"] == "CA"
    ]
    .copy()
    .sort_values("data_year")
)

# Population measures
population_cols = {
    "population": "Total population",
    "population_65_plus": "Age 65+",
    "population_80_plus": "Age 80+",
    "population_85_plus": "Age 85+"
}

# Index each series to 100 in the first common year
for col in population_cols:
    canada[f"{col}_index"] = (
        canada[col] / canada[col].iloc[0]
    ) * 100

# Plot
fig, ax = plt.subplots(figsize=(10, 5.5))

for col, label in population_cols.items():
    ax.plot(
        canada["data_year"],
        canada[f"{col}_index"],
        linewidth=2,
        label=label
    )

ax.axhline(
    100,
    linewidth=0.8,
    linestyle="--"
)

ax.set_title(
    "Population Growth and Demographic Aging in Canada (1971-2025)",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Year")
ax.set_ylabel("Population Index (First Year = 100)")

ax.legend(
    frameon=False,
    ncol=2
)

ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()
plt.show()

<table width="100%">
<tr>
<td width="50%" align="left"><a href="./00_EDA.ipynb">← Previous: EDA</a></td>
<td width="50%" align="right"><a href="./02_RQ2_capacity_workforce.ipynb">Next: RQ2 — Capacity Workforce →</a></td>
</tr>
</table>

> ##### **Insights:** 
> - ##### The data suggests that Canadian demographic ageing pressue varies across provinces. Some Atlantic provinces and Saskatchewan have relatively high concentrations of older population, while some territories experienced faster growth in older age group from a smaller population bases. At the national level, 80 plus and 85 plus populations are growing faster compared to 65 plus population. As a result, healthcare demand will likely evolve through population growth and changes in the age composition of the population.